## Notebook 207: does the HP advantage track selection specifically, or just divergence?

Split out from [notebook 206](./206_dnds_hp_alphabet_mechanism.ipynb)'s section 8 &mdash; 206 established that the HP-vs-protein/dayhoff AUC gap tracks percent-identity divergence on curated family/MHC anchors (antiviral restriction factors, MHC class I). But percent identity can't tell apart two different reasons a gene pair has diverged: **relaxed constraint** (drift under low selective pressure) and **positive selection** (active adaptive change, e.g. host-pathogen arms races at antiviral restriction factors or MHC). Both produce the identical k-mer-mismatch problem 206's mechanism is about, so percent identity alone is the right covariate for *that* claim &mdash; but it can't show whether HP's advantage is specifically a *selection* signature, which is a stronger and more interesting claim (and the one that would make antiviral restriction factors and MHC class I more than coincidentally the two strongest anchors in 206's reconciliation ledger).

**The separating experiment:** compute real per-gene &omega; (dN/dS) and regress the HP-vs-protein gap on &omega; *with percent identity as a covariate*. If &omega;'s coefficient survives conditioning on percent identity, the HP advantage tracks selection specifically, not just divergence &mdash; a mechanistic result, not a benchmark observation. If it doesn't survive, this is a divergence result and should be reported as such, not oversold as a selection result.

**Scope, stated before any pipeline code below runs:** only **pairwise &omega;** (one value per gene, from the human-mouse CDS pair) is computed here &mdash; the same quantity Ensembl used to serve genome-wide before discontinuing it at Release 100. Site-level tests (PAML M7-vs-M8, HyPhy MEME/FUBAR) need a multi-taxa phylogeny for statistical power; a single human-mouse pair can't support them, so they're out of scope here too. Extending to site-level selection would mean pulling in CDS for several more species and building gene trees &mdash; a separate, larger project.

**Target gene list** (family/MHC anchors + a percent-identity-stratified sample across 206's deciles) is built and cached in 206's own section 8 (`206_omega_gene_pairs.csv`) &mdash; that step depends on 206's in-memory `FAMILIES`/decile machinery, so it stayed there rather than being duplicated here; this notebook picks up from the cached CSV.

**This work happens outside the notebook**, in a new containerized Nextflow pipeline: [`nextflow-runs/human-mouse-dnds-omega/`](../nextflow-runs/human-mouse-dnds-omega/) (fetches CDS from Ensembl REST, aligns proteins with MAFFT, back-translates to a codon alignment, runs PAML `codeml` pairwise). **It has not been run** &mdash; PAML isn't installed anywhere reachable while this was written, so the pipeline is written and its DSL2/Groovy wiring smoke-tested, but the `codeml` output parser has not been checked against a real PAML run. The cells below (a) load the gene list 206 wrote, (b) hand off the pipeline command, and (c) will run the regression once the pipeline's output exists &mdash; until then they print what's missing rather than fabricating a result.


In [1]:
import sys
sys.path.insert(0, ".")

from pathlib import Path

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

import ortholog_analysis_utils as u

DATA_DIR = u.DATA_DIR
OUT_DIR = Path(".")

mgi_pairs, mgi_set = u.load_mgi_orthologs()

# Established best-k (notebook 200) and the 7 HP variants swept in 206 -- same constants,
# redefined here rather than re-running 206's whole sweep just to get two ints and a list.
BEST_PROTEIN_K, BEST_DAYHOFF_K = 15, 20
ALL_HP_ENCODINGS = [
    "hp", "hp-lehninger", "hp-thomas-dill", "hp-kyte-doolittle",
    "hp-thomas-dill-no-c", "hp-lehninger-plus-c", "hp-pbotc-1st-ed",
]

OMEGA_GENE_PAIRS_CSV = DATA_DIR / "206_omega_gene_pairs.csv"
if not OMEGA_GENE_PAIRS_CSV.exists():
    print(f"{OMEGA_GENE_PAIRS_CSV} not found -- run notebook 206's section 8 gene-list-export "
          "cell first (builds and caches the target gene list this notebook consumes).")
    omega_gene_pairs_df = pl.DataFrame()
else:
    omega_gene_pairs_df = pl.read_csv(str(OMEGA_GENE_PAIRS_CSV))
    print(f"Loaded {omega_gene_pairs_df.height:,} target gene pairs from {OMEGA_GENE_PAIRS_CSV}")
    print(omega_gene_pairs_df.group_by("source").len().sort("len", descending=True))


Loaded 1,338 target gene pairs from /Users/olga/data/gencode/results-human-mouse-orthologs/206_omega_gene_pairs.csv
shape: (10, 2)
┌─────────────────────────────────┬─────┐
│ source                          ┆ len │
│ ---                             ┆ --- │
│ str                             ┆ u32 │
╞═════════════════════════════════╪═════╡
│ Sperm/testis-specific (Kopania… ┆ 491 │
│ genome_wide_stratified          ┆ 475 │
│ Olfactory receptors             ┆ 323 │
│ CYP2/CYP3                       ┆ 28  │
│ Antiviral restriction factors   ┆ 9   │
│ MHC II alpha                    ┆ 4   │
│ MHC processing (ctrl)           ┆ 3   │
│ MHC II beta                     ┆ 2   │
│ MHC I non-classical             ┆ 2   │
│ MHC I classical                 ┆ 1   │
└─────────────────────────────────┴─────┘


In [2]:
# Hand-off: this is a new tool install (PAML, via a container — nothing installed on the
# host) plus a real network fetch (Ensembl REST, hundreds of genes) plus non-trivial compute
# (MAFFT + codeml per gene) — run it yourself and watch the first (`make test`) invocation
# especially closely, since the codeml output parser hasn't been checked against real PAML
# output yet (see section 8 intro above).
print("Run the omega pipeline (not run automatically from this notebook):\n")
print("  cd ../nextflow-runs/human-mouse-dnds-omega")
print("  make test    # 5 gene pairs — validates the codeml .mlc parser before scaling up")
print("  make run     # full gene set from the CSV just written\n")
print(f"Input just written: {OMEGA_GENE_PAIRS_CSV}")
print("Expected output: nextflow-runs/human-mouse-dnds-omega/results/omega_results.tsv")
print("  columns: human_gene, mouse_gene, dN, dS, omega, t, N, S, n_codons, status, detail")
print("\nOnce that file exists, re-run the cells below — they'll pick it up automatically.")


Run the omega pipeline (not run automatically from this notebook):

  cd ../nextflow-runs/human-mouse-dnds-omega
  make test    # 5 gene pairs — validates the codeml .mlc parser before scaling up
  make run     # full gene set from the CSV just written

Input just written: /Users/olga/data/gencode/results-human-mouse-orthologs/206_omega_gene_pairs.csv
Expected output: nextflow-runs/human-mouse-dnds-omega/results/omega_results.tsv
  columns: human_gene, mouse_gene, dN, dS, omega, t, N, S, n_codons, status, detail

Once that file exists, re-run the cells below — they'll pick it up automatically.


In [3]:
OMEGA_RESULTS_TSV = Path("../nextflow-runs/human-mouse-dnds-omega/results/omega_results.tsv")
OMEGA_SCORES_CSV = DATA_DIR / "206_omega_gene_true_pair_scores.csv"
OMEGA_TARGET_ALPHABETS = [("protein", BEST_PROTEIN_K), ("dayhoff", BEST_DAYHOFF_K)] + [(enc, 30) for enc in ALL_HP_ENCODINGS]

if not OMEGA_RESULTS_TSV.exists():
    print(f"{OMEGA_RESULTS_TSV} not found — pipeline hasn't been run yet (see hand-off above).")
    print("Nothing to regress. Re-run this cell after `make run` finishes.")
else:
    omega_df = pl.read_csv(str(OMEGA_RESULTS_TSV), separator="\t").filter(pl.col("status") == "ok")
    print(f"Loaded real ω for {omega_df.height}/{omega_gene_pairs_df.height} attempted gene pairs "
          f"({100 * omega_df.height / omega_gene_pairs_df.height:.0f}% success rate)")

    # Per-gene score for the gene's OWN true MGI pair specifically (not a family-wide AUC —
    # AUC needs multiple pairs, this regression needs one number per gene). Reuses
    # u.load_families_kmerseek_scores exactly as section 4 does (one raw-file scan per
    # combo, not per gene), treating the omega target set as a single pseudo-family.
    if OMEGA_SCORES_CSV.exists():
        gene_scores_df = pl.read_csv(str(OMEGA_SCORES_CSV))
    else:
        target_family = {"omega_targets": set(omega_gene_pairs_df["human_gene"].to_list())}
        score_rows = []
        for enc, k in OMEGA_TARGET_ALPHABETS:
            try:
                dfs = u.load_families_kmerseek_scores(enc, k, target_family, mgi_ortholog_set=mgi_set)
            except FileNotFoundError:
                print(f"{enc:20s} k={k:<3d} MISSING genome-wide file, skipping")
                continue
            df = dfs.get("omega_targets")
            if df is None or df.height == 0:
                continue
            true_pairs = df.filter(pl.col("label") == 1)
            for row in true_pairs.iter_rows(named=True):
                score_rows.append({"human_gene": row["human_gene"], "mouse_gene": row["mouse_gene"],
                                    "encoding": enc, "ksize": k, "score": row["score_bonf_neglogp_cont"]})
            print(f"{enc:20s} k={k:<3d} {true_pairs.height} true pairs scored")
        gene_scores_df = pl.DataFrame(score_rows)
        gene_scores_df.write_csv(str(OMEGA_SCORES_CSV))

    # HP score per gene = median across the 6 variants (same envelope convention used
    # throughout this notebook), so one variant's idiosyncrasy doesn't drive the regression.
    hp_gene_score = (
        gene_scores_df.filter(pl.col("encoding").is_in(ALL_HP_ENCODINGS))
        .group_by("human_gene").agg(pl.col("score").median().alias("hp_score"))
    )
    protein_gene_score = gene_scores_df.filter(pl.col("encoding") == "protein").select(
        "human_gene", pl.col("score").alias("protein_score")
    )

    regression_df = (
        omega_gene_pairs_df
        .join(omega_df.select(["human_gene", "mouse_gene", "omega"]), on=["human_gene", "mouse_gene"], how="inner")
        .join(hp_gene_score, on="human_gene", how="inner")
        .join(protein_gene_score, on="human_gene", how="inner")
        .with_columns((pl.col("hp_score") - pl.col("protein_score")).alias("gap"))
        .drop_nulls(["omega", "perc_id", "gap"])
    )
    print(f"\nRegression dataset: n={regression_df.height} genes with ω + perc_id + score-gap all present")
    print(regression_df.select(["human_gene", "source", "perc_id", "omega", "gap"]).head(10))


../nextflow-runs/human-mouse-dnds-omega/results/omega_results.tsv not found — pipeline hasn't been run yet (see hand-off above).
Nothing to regress. Re-run this cell after `make run` finishes.


In [4]:
import statsmodels.api as sm

if not OMEGA_RESULTS_TSV.exists():
    print("Skipping regression — pipeline output not present yet (see above).")
elif regression_df.height < 15:
    print(f"n={regression_df.height} is below this notebook's own n-floor (15, see sections 4/6) — "
          "not reporting a regression on too few genes.")
else:
    X = sm.add_constant(regression_df.select(["omega", "perc_id"]).to_numpy())
    y = regression_df["gap"].to_numpy()
    model = sm.OLS(y, X, missing="drop").fit()
    print(model.summary(xname=["const", "omega", "perc_id"]))

    omega_coef, omega_p = model.params[1], model.pvalues[1]
    print()
    if omega_p < 0.05:
        print(f"omega SURVIVES controlling for percent identity (coef={omega_coef:+.4f}, p={omega_p:.4f}): "
              "consistent with H1 -- the HP advantage tracks selection specifically, not just divergence.")
    else:
        print(f"omega does NOT survive controlling for percent identity (coef={omega_coef:+.4f}, p={omega_p:.4f}): "
              "this is a divergence result, not (yet) a selection result -- report it as such, per the "
              "notebook intro's own framing.")


Skipping regression — pipeline output not present yet (see above).


### Reading this once it's run

- **The outcome variable (`gap`) is per-gene, not the family-bucketed AUC gap used in sections 4-7.** It's each gene's own true-MGI-pair score under HP (median of the 6 variants) minus under protein — the only per-gene analog available, since AUC needs a set of pairs. Don't conflate the sign/magnitude here with the AUC gaps earlier in the notebook; they're related but not the same quantity.
- **The regression is the actual test, not the raw ω values.** A positive, significant ω coefficient *after* percent identity is already in the model is the H1 signature. If ω only looks predictive on its own (no `perc_id` term) that's not informative — percent identity and ω are correlated by construction (both measure divergence), so the whole point of the two-covariate model is to ask whether ω adds anything *beyond* what percent identity already explains.
- **n depends on two things outside this notebook's control**: the Ensembl REST fetch success rate (canonical transcripts aren't guaranteed for every gene, especially in the family anchors, which include some non-classical/immune genes with messier annotation), and the codeml parser actually working (unvalidated as of this writing — see the section 8 intro). If n comes back much smaller than the ~hundreds targeted, that's the first thing to check, not the regression result.